# Extension 05 Elastic and wrinkling nets

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**Once a cable net or textile membrane is fabricated with fixed rest lengths $l_0$ and axial stiffness $EA$, how does it deform, wrinkle, or lock up under new loads?**

Inspired by the working solvers in `damianbrenlla/form-finding` (`python_core/solvers_ff.py`: `ForceDensitySolver` and `UnderwoodDRSolver`) and `JonasFeron/Muscle` (`GH_BilinearMaterial.cs` and `nonlinear_dm.py`).

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** in the presentation's materialisation stage, designing a state $(l_k, f_k)$ specifies each bar's unstretched cut length:

$$
l_{0,k} = \frac{EA_k\,l_k}{EA_k + f_k}
$$

When new asymmetric loads, support settlements, or wind uplift act on the built structure, $l_{0,k}$ and $EA_k$ are fixed physical constants while $q_k(l_k) = f_k(l_k)/l_k$ becomes state dependent! For a tension only cable or fabric yarn that goes slack under compression (`np.maximum(0.0, ...)` in `solvers_ff.py`):

$$
f_k(l_k) = \max\!\left(0,\; EA_k\,\frac{l_k - l_{0,k}}{l_{0,k}}\right), \qquad q_k(l_k) = \max\!\left(q_{\text{floor}},\; \frac{f_k(l_k)}{l_k}\right)
$$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_1d.svg"
       style="width:100%; height:auto;"
       alt="Elastic bar stretching from unstrained rest length l0 to equilibrium length l" />
  <p style="text-align:center;">
    <em>Updating q(l) = f(l) / l from fixed rest lengths l<sub>0</sub> captures elastic load redistribution, slack wrinkling, and warp weft anisotropy.</em>
  </p>
</div>

- Cut exact fabrication rest lengths $l_0$ from a symmetric saddle design and solve true elastic deflection under an eccentric snow patch
- Capture tension only cable slackening and membrane wrinkling under upward wind uplift (`solvers_ff.py` and `Muscle`)
- Model anisotropic warp and weft textile stiffness, stiff perimeter edge cables, and a shortened knitted rib channel

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# From Design Force Densities to Fixed Rest Lengths $l_0$

First we form find a symmetric hypar saddle roof ($z = 0\text{ m}$ at $n_0, n_{48}$ and $z = 2.2\text{ m}$ at $n_6, n_{42}$) with $q_{\text{design}} = 2.5\text{ kN/m}$ inside and $8.0\text{ kN/m}$ on the four boundary edge cables, then freeze its unstretched cut lengths $l_{0,k} = \frac{EA_k\,l_k}{EA_k + f_k}$. Next we apply an eccentric snow patch load ${\color{#2e9e4a} p_z = -0.85\text{ kN}}$ on one quadrant and iterate `# equation 6.32` with damped updates of $q_k(l_k)$ until the true Hookean residual vanishes.

In [1]:
import numpy as np
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.76, 0.76, 0.76, 1.0)

# 6 x 6 grid (49 nodes, 84 bars)
n = 7
gx, gy = np.meshgrid(range(n), range(n))
x0, y0 = gx.flatten().astype(float), gy.flatten().astype(float)
z0 = np.zeros(n * n)

edges = [(k, k + 1) for k in range(n * n) if x0[k] < n - 1] + [
    (k, k + n) for k in range(n * n) if y0[k] < n - 1
]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1.0, -1.0

fixed = [0, n - 1, n * (n - 1), n * n - 1]
free = [k for k in range(n * n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# hypar corner elevations: (0,0)->0, (6,0)->2.2, (0,6)->2.2, (6,6)->0
xyz_des = np.column_stack([x0.copy(), y0.copy(), z0.copy()])
xyz_des[n - 1, 2] = 2.2
xyz_des[n * (n - 1), 2] = 2.2

is_bnd_edge = np.array([
    (x0[a] in (0.0, 6.0) and x0[b] == x0[a]) or (y0[a] in (0.0, 6.0) and y0[b] == y0[a])
    for a, b in edges
])
q_des = np.where(is_bnd_edge, 8.0, 2.5)
Q_des = np.diag(q_des)

# equation 6.32
A_des = C_N.T @ Q_des @ C_N
for axis in range(3):
    xyz_des[free, axis] = np.linalg.solve(A_des, -C_N.T @ Q_des @ C_F @ xyz_des[fixed, axis])

l_des = np.linalg.norm(C @ xyz_des, axis=1)
f_des = q_des * l_des

# assign axial stiffness EA (kN) and freeze fabrication rest lengths l_0 (presentation eq. 6.37)
EA = np.where(is_bnd_edge, 120.0, 35.0)
l_0 = EA * l_des / (EA + f_des)


def solve_elastic_fdm(xyz_seed, C_free, C_fix, free_idx, fix_idx, l_rest, EA_vec, p_loads, max_iter=600):
    """Iterative nonlinear FDM updating q(l) = f(l)/l from fixed rest lengths l_rest."""
    xyz = xyz_seed.copy()
    l_cur = np.linalg.norm(C @ xyz, axis=1)
    q_cur = np.maximum(EA_vec * (l_cur - l_rest) / (l_rest * l_cur), 1e-4)

    for it in range(max_iter):
        Q_mat = np.diag(q_cur)
        A_mat = C_free.T @ Q_mat @ C_free
        for axis in range(3):
            xyz[free_idx, axis] = np.linalg.solve(
                A_mat, p_loads[free_idx, axis] - C_free.T @ Q_mat @ C_fix @ xyz[fix_idx, axis]
            )

        vec = C @ xyz
        l_cur = np.linalg.norm(vec, axis=1)
        strain = (l_cur - l_rest) / l_rest
        f_phys = np.maximum(0.0, EA_vec * strain)
        q_target = np.maximum(f_phys / l_cur, 1e-6)

        # check true physical equilibrium residual at free nodes using tension only f_phys
        r_free = p_loads[free_idx] - C_free.T @ ((f_phys / l_cur)[:, None] * vec)
        if np.abs(r_free).max() < 1e-7:
            break
        omega = np.where(strain <= 0.0, 0.35, 0.8 / (1.0 + EA_vec / np.maximum(q_cur * l_cur, 0.1)))
        q_cur = (1.0 - omega) * q_cur + omega * q_target

    return xyz, l_cur, f_phys, np.abs(r_free).max(), it + 1


# 1. verify zero load recovers exact design geometry; 2. apply eccentric snow load on upper right quadrant
xyz_check, _, f_check, res_check, _ = solve_elastic_fdm(xyz_des, C_N, C_F, free, fixed, l_0, EA, np.zeros((n * n, 3)))
print("design recovery error ||xyz - xyz_des|| =", f"{np.abs(xyz_check - xyz_des).max():.2e}", "m")

p_snow = np.zeros((n * n, 3))
snow_nodes = [k for k in free if x0[k] >= 3.0 and y0[k] >= 3.0]
p_snow[snow_nodes, 2] = -0.85

# compare naive constant q_des linear prediction vs true nonlinear elastic cable net with fixed l_0
xyz_naive = xyz_des.copy()
for axis in range(3):
    xyz_naive[free, axis] = np.linalg.solve(A_des, p_snow[free, axis] - C_N.T @ Q_des @ C_F @ xyz_naive[fixed, axis])

xyz_snow, l_snow, f_snow, res_snow, it_snow = solve_elastic_fdm(xyz_des, C_N, C_F, free, fixed, l_0, EA, p_snow)
probe = 32  # node at (4, 4) inside snow patch
print("elastic snow iterations =", it_snow, " max physical residual =", f"{res_snow:.2e}", "kN")
print("node n32 z: design =", round(xyz_des[probe, 2], 3), "m | naive fixed q =", round(xyz_naive[probe, 2], 3), "m | true elastic EA =", round(xyz_snow[probe, 2], 3), "m")
print("peak cable force rose from", round(f_des.max(), 2), "kN to", round(f_snow.max(), 2), "kN (stiffening limits deflection!)")

colors_sn = [Color(0.65 - 0.65 * fv / f_snow.max(), 0.65 - 0.65 * fv / f_snow.max(), 0.65 - 0.65 * fv / f_snow.max(), 1.0) for fv in f_snow]
session_sn = Session("fdm_ext05_elastic_snow")
session_sn.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_des.tolist()), [GRAY] * len(edges), [1.4] * len(edges))
session_sn.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_snow.tolist()), colors_sn, [3.0] * len(edges))
for net in range(2):
    for k in range(n * n):
        pt = session_sn.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in fixed else RED
        pt.width = 13.0 if k in fixed else 6.5

for k in snow_nodes:
    lv = Line(xyz_snow[k, 0], xyz_snow[k, 1], xyz_snow[k, 2], xyz_snow[k, 0], xyz_snow[k, 1], xyz_snow[k, 2] - 0.55)
    lv.linecolor = GREEN
    lv.width = 2.4
    lv.arrowhead = Arrowhead.END
    session_sn.add_line(lv)

show(session_sn, "fdm_ext05_elastic_snow")

design recovery error ||xyz - xyz_des|| = 7.99e-15 m
elastic snow iterations = 212  max physical residual = 9.88e-08 kN
node n32 z: design = 0.99 m | naive fixed q = 0.092 m | true elastic EA = 0.285 m
peak cable force rose from 10.08 kN to 12.47 kN (stiffening limits deflection!)


# Tension Only Slackening and Membrane Wrinkling

Consider a $6\text{ m} \times 6\text{ m}$ textile panel stretched $4\%$ along $x$ ($x_{\text{frame}} = 1.04\,x_0$) and compressed $6\%$ along $y$ ($y_{\text{frame}} = 0.94\,y_0$) under upward wind uplift ${\color{#2e9e4a} p_z = +0.25\text{ kN}}$. Because fabrics and cables cannot carry compression, bars that shorten below their cut length ($l_k \le l_{0,k}$) go **slack** ($f_k = 0\text{ kN}$) following `solvers_ff.py` (`axial_forces = np.maximum(0.0, ...)`), forming parallel wrinkle bands while all wind suction transfers along the taut $x$ warp strips!

In [2]:
# perimeter frame stretched +4% in x and compressed -6% in y under upward wind suction
fixed_perim = [k for k in range(n * n) if x0[k] in (0.0, 6.0) or y0[k] in (0.0, 6.0)]
free_perim = [k for k in range(n * n) if k not in fixed_perim]
C_Np, C_Fp = C[:, free_perim], C[:, fixed_perim]

xyz_wr_seed = np.column_stack([x0 * 1.04, y0 * 0.94, np.zeros(n * n)])
l0_wr = np.full(len(edges), 0.985)
EA_wr = np.full(len(edges), 45.0)

p_wind = np.zeros((n * n, 3))
p_wind[free_perim, 2] = 0.25

xyz_wrink, l_wrink, f_wrink, res_wrink, it_wrink = solve_elastic_fdm(
    xyz_wr_seed, C_Np, C_Fp, free_perim, fixed_perim, l0_wr, EA_wr, p_wind
)
f_hooke_raw = EA_wr * (l_wrink - l0_wr) / l0_wr
slack_mask = f_wrink < 1e-3

print("wrinkling solver iterations             =", it_wrink, " max nodal residual =", f"{res_wrink:.2e}", "kN")
print("raw Hooke force without slack cutoff    =", round(f_hooke_raw.min(), 3), "kN (unphysical compression!)")
print("tension only slack cables (f = 0 kN)    =", int(slack_mask.sum()), "of", len(edges), "bars")
print("peak taut warp tension under uplift     =", round(f_wrink.max(), 2), "kN")

# render slack wrinkled bars as thin light grey lines and taut load bearing cables in dark black
colors_wr = []
widths_wr = []
for e, fv in enumerate(f_wrink):
    if slack_mask[e]:
        colors_wr.append(GRAY)
        widths_wr.append(1.0)
    else:
        shade = 0.55 - 0.55 * fv / f_wrink.max()
        colors_wr.append(Color(shade, shade, shade, 1.0))
        widths_wr.append(3.4)

session_wr = Session("fdm_ext05_wrinkling_uplift")
session_wr.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_wrink.tolist()), colors_wr, widths_wr)
for k in range(n * n):
    pt = session_wr.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_perim else RED
    pt.width = 11.0 if k in fixed_perim else 6.5

show(session_wr, "fdm_ext05_wrinkling_uplift")

wrinkling solver iterations             = 600  max nodal residual = 9.71e-07 kN
raw Hooke force without slack cutoff    = -2.056 kN (unphysical compression!)
tension only slack cables (f = 0 kN)    = 40 of 84 bars
peak taut warp tension under uplift     = 3.03 kN


# Anisotropic Warp Weft Textile with Stiff Boundary Cables and Knitted Rib

Finally we implement the directional fabric features from `UnderwoodDRSolver` in `solvers_ff.py` together with bilinear yarn lock up:

- **Anisotropy:** warp yarns along $x$ ($EA_{\text{warp}} = 28\text{ kN}$) are four times stiffer than weft yarns along $y$ ($EA_{\text{weft}} = 7\text{ kN}$), surrounded by stiff steel perimeter edge cables ($EA_{\text{edge}} = 140\text{ kN}$),
- **Loop lock up:** once textile strain $\varepsilon_k = (l_k - l_{0,k})/l_{0,k}$ exceeds $\varepsilon_{\text{lock}} = 0.08$, yarn interlooping jammed stiffness triples ($EA_2 = 3\,EA_1$),
- **Shortened rib channel:** shortening $l_0$ along the central row $y = 3\text{ m}$ pulls an upward stiffening corrugation into a wet concrete shell formwork!

In [3]:
is_warp = np.array([abs(x0[b] - x0[a]) >= abs(y0[b] - y0[a]) for a, b in edges])
is_rib = np.array([y0[a] == 3.0 and y0[b] == 3.0 for a, b in edges])

EA_1 = np.where(is_bnd_edge, 140.0, np.where(is_warp, 28.0, 7.0))
EA_1[is_rib] = 75.0
EA_2 = 3.0 * EA_1
eps_lock = 0.08

# flat 6 x 6 textile sheet pre tensioned to rest lengths 0.94 m (0.68 m along central rib channel)
xyz_tex = np.column_stack([x0.copy(), y0.copy(), np.zeros(n * n)])
l0_tex = np.where(is_bnd_edge, 0.96, 0.94)
l0_tex[is_rib] = 0.68
# elevate central rib anchors on left and right boundaries to form an arched rib channel
xyz_tex[21, 2] = 0.9
xyz_tex[27, 2] = 0.9
fixed_tex = sorted(set(fixed + [21, 27]))
free_tex = [k for k in range(n * n) if k not in fixed_tex]
C_Nt, C_Ft = C[:, free_tex], C[:, fixed_tex]

p_wet = np.zeros((n * n, 3))
p_wet[free_tex, 2] = -0.22

q_tex = np.full(len(edges), 2.5)
for it_t in range(600):
    Q_t = np.diag(q_tex)

    # equation 6.32
    A_t = C_Nt.T @ Q_t @ C_Nt
    for axis in range(3):
        xyz_tex[free_tex, axis] = np.linalg.solve(
            A_t, p_wet[free_tex, axis] - C_Nt.T @ Q_t @ C_Ft @ xyz_tex[fixed_tex, axis]
        )

    vec_t = C @ xyz_tex
    l_t = np.linalg.norm(vec_t, axis=1)
    strain = (l_t - l0_tex) / l0_tex
    f_t = np.where(
        strain <= 0.0,
        0.0,
        np.where(strain <= eps_lock, EA_1 * strain, EA_1 * eps_lock + EA_2 * (strain - eps_lock)),
    )
    EA_tan = np.where(strain <= 0.0, 0.0, np.where(strain <= eps_lock, EA_1, EA_2))
    r_ft = p_wet[free_tex] - C_Nt.T @ ((f_t / l_t)[:, None] * vec_t)
    if np.abs(r_ft).max() < 1e-7:
        break
    q_targ = np.maximum(f_t / l_t, 1e-6)
    omega = np.where(strain <= 0.0, 0.35, 0.8 / (1.0 + EA_tan / np.maximum(q_tex * l_t, 0.1)))
    q_tex = (1.0 - omega) * q_tex + omega * q_targ

locked_bars = int(np.sum(strain > eps_lock))
print("anisotropic textile iterations =", it_t + 1, " max residual =", f"{np.abs(r_ft).max():.2e}", "kN")
print("bars in jammed lock up regime  =", locked_bars, "bars")
print("rib crown z24 =", round(xyz_tex[24, 2], 3), "m  vs side bay sag z10 =", round(xyz_tex[10, 2], 3), "m")

colors_tx = [Color(0.65 - 0.65 * fv / f_t.max(), 0.65 - 0.65 * fv / f_t.max(), 0.65 - 0.65 * fv / f_t.max(), 1.0) for fv in f_t]
widths_tx = [4.2 if (is_bnd_edge[e] or is_rib[e]) else 2.1 for e in range(len(edges))]

session_tx = Session("fdm_ext05_anisotropic_textile")
session_tx.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_tex.tolist()), colors_tx, widths_tx)
for k in range(n * n):
    pt = session_tx.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_tex else RED
    pt.width = 13.0 if k in fixed_tex else 6.5

show(session_tx, "fdm_ext05_anisotropic_textile")

anisotropic textile iterations = 563  max residual = 9.82e-08 kN
bars in jammed lock up regime  = 26 bars
rib crown z24 = 0.861 m  vs side bay sag z10 = -0.062 m


# Observation

**Coupling `# equation 6.32` with the secant update $q_k(l_k) = f_k(l_k)/l_k$ closes the loop between geometric form finding and physical structural analysis.** Once fabrication rest lengths $l_0$ are cut, the same matrix equation predicts nonlinear cable stiffening under eccentric snow, zero compression wrinkling zones under wind uplift, and directional warp weft shaping in knitted shell formworks.